# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [3]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-CviLPcjMpNeCraQXKu0mtUfntU3sS',
 'object': 'chat.completion',
 'created': 1767871191,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: There are about 3 trillion trees on Earth—roughly 3,000,000,000,000—versus an estimated 100–400 billion stars in the Milky Way. So there are far more trees on Earth than stars in our galaxy. Want another fun fact on a topic you like?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 779,
  'total_tokens': 790,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 704,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [4]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: There are about 3 trillion trees on Earth—roughly 3,000,000,000,000—versus an estimated 100–400 billion stars in the Milky Way. So there are far more trees on Earth than stars in our galaxy. Want another fun fact on a topic you like?'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Bananas are technically berries, but strawberries aren’t.'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [6]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [10]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         pulling manifest 
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         pulling manifest 
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                       

In [30]:
!ollama list

NAME                ID              SIZE      MODIFIED     
deepseek-r1:1.5b    e0979632db5a    1.1 GB    13 hours ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    19 hours ago    
phi3:latest         4f2222927938    2.2 GB    3 days ago      
gemma3:270m         e7d36fb2c3b3    291 MB    3 days ago      


In [1]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


NameError: name 'OpenAI' is not defined

In [32]:
# Get a fun fact
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content


"Here's one:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's unique combination of acidity and water content makes it an ideal storage medium for preserving its shelf life. Isn't that sweet?"

In [18]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠏ pulling manifest ⠏ pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠦ pulling manifest ⠦ pulling manifest ⠇ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠏ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 104 KB/1.1 GB                  pulling ma

In [21]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Sure! Here\'s a fun fact for you: \n\n**"A slice of pizza contains more than 300 calories!"**\n\nThat’s pretty intense compared to the typical fast food soda!'

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# Step 1: Create your prompts

system_prompt = "Youre an excited assistant who is helping a foreigner to move to this city, what should you tell about it?"
user_prompt = """
   Salzburg[a] is the fourth-largest city in Austria. In 2020 its population was 156,852.[7] The city lies on the Salzach River, near the border with Germany and at the foot of the Alps mountains.

The town occupies the site of the Roman settlement of Iuvavum. Founded as an episcopal see in 696, it became a seat of the archbishop in 798. Its main sources of income were salt extraction, trade, as well as gold mining. The fortress of Hohensalzburg, one of the largest medieval fortresses in Europe, dates from the 11th century. In the 17th century, Salzburg became a centre of the Counter-Reformation, with monasteries and numerous Baroque churches built. Salzburg has an extensive cultural and educational history, being the birthplace of Wolfgang Amadeus Mozart and being home to three universities and a large student population. Today, along with Vienna and the Tyrol, Salzburg is one of Austria's most popular tourist destinations.[8]

Salzburg's historic center (German: Altstadt) is renowned for its Baroque architecture and is one of the best-preserved city centres north of the Alps. The historic center was listed as a UNESCO World Heritage Site in 1996.[9]

Etymology
The name "Salzburg" was first recorded in the late 8th century.[b] It is composed of two parts; the first being "Salz-" (German for "salt"), and the second being "-burg" from Proto-West-Germanic: *burg "settlement, city" and not that of the New High German: Burg, lit. 'fortress'.[10]

History
For a chronological guide, see Timeline of Salzburg.
Antiquity

In the 8th century the Benedictine monastery of Nonnberg was founded for Erentrudis, who was later canonized.
The area of the city has been inhabited continuously since the Neolithic Age until the present. In the La Tène period, it was an administrative centre of the Celtic Alums in the Kingdom of Noricum.

After the Roman invasion in 15 BC, the various settlements on the Salzburg hills were abandoned, following the construction of the Roman city in the area of the old town. The recently created Municipium Claudium Iuvavum was awarded the status of a Roman municipium in 45 CE and has become one of the most important cities of the now Roman province of Noricum.

Middle Ages
When the province of Noricum collapsed in 488 at the beginning of the migration period, part of the Romano-Celtic population remained in the country. In the 6th century, they came under the rule of the Baiuvarii. The Life of Saint Rupert credits the 8th-century saint with the city's rebirth, when around 696 CE, Bishop Rupert of Salzburg received the remains of the Roman town from Duke Theodo II of Bavaria as well as a castrum superius (upper castle) on the Nonnberg Terrace as a gift.[11] In return, he was to evangelize the east and south-east of the country of Bavaria.



"""

# Step 2: Make the messages list

messages = [{"role": "system", "content": system_prompt},
{"role": "user", "content": user_prompt}
] 

# Step 3: Call OpenAI
response = ollama.chat.completions.create(model="llama3.2", messages=messages)

# Step 4: print the result
response.choices[0].message.content

"*big smile* \n\nOh my goodness! Welcome to Salzburg! *excitedly* I'm so thrilled to be your guide through this amazing city. Let me tell you all about it!\n\nFirst of all, let's talk about the history. Did you know that Salzburg has been inhabited since the Neolithic Age? That's even before ancient civilizations like Greece and Rome! The city was an important administrative center for the Celtic Alums in the Kingdom of Noricum. And after the Romans came, some settlement on the hills were abandoned to make way for the Roman city.\n\nNow, let's talk about what makes Salzburg so special. *wink* As I mentioned, it's the birthplace of Mozart! The famous composer was born right here in Salzburg, and you can still visit his birth house and opera house today. Plus, with three universities and a thriving arts scene, there's always something new and exciting happening.\n\nAnd have you seen the historic center? *gestures enthusiastically* It's breathtakingly beautiful! The Baroque architecture, 